# Heterogeneous-Voxel At-Scale τ-p Demo (Oceanic Crust, Riccati Interlayer)

End-to-end demo of the heterogeneous-voxel block Foldy-Lax inner
solver composed with the Riccati interlayer-multiple-scattering outer
sweep, on a realistic 20-layer oceanic-crust model.  Each of the 18
finite elastic layers carries an independent 32 × 32 lattice of
cubic voxels whose material contrasts are drawn from a per-layer
Gaussian mixture (soft + hard component).

Pipeline:

1. Load the YAML config and build the `LayerModel`.
2. Sample per-voxel `MaterialContrast`s per elastic layer.
3. Per-layer inner solve (frequency-batched block GMRES on
   `(I − G · T_per_cube)`) → one composite `(F, 9, 9)` per layer.
4. Per-ω outer Riccati sweep → `R_total(ω, p)` and `R_ref(ω, p)`.
5. Ricker × Hermitian IFFT → total and reference τ-p traces.
6. Display reference and total wiggle gathers + an $|R(\omega,p)|$
   heatmap.
7. Save NPZ + PDFs under `GlobalMatrix/output/taup_scale_demo/`.

**Output-artifact locations** — `GlobalMatrix/output/taup_scale_demo/`
(NPZ, three PDFs).


In [ ]:
import sys
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

ms_root = Path("/Users/tod/Desktop/MultipleScatteringCalculations")
if str(ms_root) not in sys.path:
    sys.path.insert(0, str(ms_root))

from GlobalMatrix.taup_scale_demo import (  # noqa: E402
    build_oceanic_crust_model,
    compute_R_omega_p_stack,
    compute_layer_composite_tmatrices_het,
    load_scale_demo_config,
    plot_r_omega_p_heatmap,
    plot_taup_wiggle_gather,
    sample_voxel_contrasts,
    taup_traces_from_R_omega_p,
)

CONFIG_PATH = Path("taup_scale_demo_config.yaml")
print(f"config: {CONFIG_PATH.resolve()}")

## 1. Load YAML + Build Layer Model


In [2]:
config = load_scale_demo_config(CONFIG_PATH)
model = build_oceanic_crust_model(config)

print(f"n_layers   = {config.n_layers}")
print(f"elastic ifaces = {config.elastic_iface_range}")
print(
    f"voxels     = {config.voxels.M}×{config.voxels.M}×{config.voxels.N_z} "
    f"(a = {config.voxels.a_km * 1000:.1f} m)"
)
print(f"n_cubes/layer = {config.voxels.M**2 * config.voxels.N_z}")
print(f"nw         = {config.nw}  (positive frequencies)")
print(f"T_record   = {config.T_record:.1f} s")
print(f"damping    = {config.damping:g} rad/s")
print(f"slowness   = [{config.p_min:.3f}, {config.p_max:.3f}] s/km, n_p={config.n_p}")
print()
print("Layer stack:")
for i, layer in enumerate(config.layers):
    thick_str = (
        "  half-space"
        if not np.isfinite(layer.thickness)
        else f"{layer.thickness:6.3f} km"
    )
    print(
        f"  [{i:2d}] {layer.name:>10s}  α={layer.alpha:4.2f}  β={layer.beta:4.2f}  "
        f"ρ={layer.rho:4.2f}  thick={thick_str}"
    )

n_layers   = 20
elastic ifaces = (1, 18)
voxels     = 32×32×1 (a = 10.0 m)
n_cubes/layer = 1024
nw         = 256  (positive frequencies)
T_record   = 32.0 s
damping    = 0.1 rad/s
slowness   = [0.050, 0.600] s/km, n_p=48

Layer stack:
  [ 0]      water  α=1.50  β=0.00  ρ=1.03  thick= 4.500 km
  [ 1] sediment_1  α=1.80  β=0.40  ρ=1.80  thick= 0.300 km
  [ 2] sediment_2  α=2.10  β=0.70  ρ=2.00  thick= 0.300 km
  [ 3] sediment_3  α=2.40  β=1.00  ρ=2.20  thick= 0.300 km
  [ 4]   pillow_1  α=3.50  β=1.90  ρ=2.50  thick= 0.500 km
  [ 5]   pillow_2  α=4.00  β=2.20  ρ=2.60  thick= 0.500 km
  [ 6]   pillow_3  α=4.50  β=2.50  ρ=2.70  thick= 0.500 km
  [ 7]     dyke_1  α=5.00  β=2.80  ρ=2.75  thick= 0.500 km
  [ 8]     dyke_2  α=5.30  β=2.95  ρ=2.80  thick= 0.500 km
  [ 9]     dyke_3  α=5.50  β=3.05  ρ=2.85  thick= 0.500 km
  [10]     dyke_4  α=5.70  β=3.15  ρ=2.88  thick= 0.500 km
  [11]   gabbro_1  α=6.50  β=3.60  ρ=2.90  thick= 0.400 km
  [12]   gabbro_2  α=6.60  β=3.65  ρ=2.92  thick= 0.400 k

## 2. Sample Per-Voxel Contrasts

Each of the 18 elastic layers gets its own 1024-long list of
`MaterialContrast` samples drawn from the layer's GMM.


In [3]:
rng = np.random.default_rng(config.random_seed)
voxel_contrasts = sample_voxel_contrasts(config, rng)

print(f"elastic ifaces sampled: {sorted(voxel_contrasts.keys())}")
print()
print("per-layer mean Δλ / Δμ / Δρ (realised sample means):")
for iface in sorted(voxel_contrasts.keys()):
    cs = voxel_contrasts[iface]
    dl = np.mean([c.Dlambda for c in cs])
    dm = np.mean([c.Dmu for c in cs])
    dr = np.mean([c.Drho for c in cs])
    layer_name = config.layers[iface].name
    print(
        f"  iface {iface:2d} ({layer_name:>10s}): "
        f"Δλ̄={dl:+.3f}  Δμ̄={dm:+.3f}  Δρ̄={dr:+.3f}"
    )

elastic ifaces sampled: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]

per-layer mean Δλ / Δμ / Δρ (realised sample means):
  iface  1 (sediment_1): Δλ̄=+0.190  Δμ̄=+0.011  Δρ̄=+0.023
  iface  2 (sediment_2): Δλ̄=+0.240  Δμ̄=+0.034  Δρ̄=+0.021
  iface  3 (sediment_3): Δλ̄=+0.404  Δμ̄=+0.124  Δρ̄=+0.044
  iface  4 (  pillow_1): Δλ̄=+0.726  Δμ̄=+0.656  Δρ̄=+0.065
  iface  5 (  pillow_2): Δλ̄=+0.999  Δμ̄=+0.923  Δρ̄=+0.070
  iface  6 (  pillow_3): Δλ̄=+1.241  Δμ̄=+1.236  Δρ̄=+0.074
  iface  7 (    dyke_1): Δλ̄=+1.564  Δμ̄=+1.646  Δρ̄=+0.077
  iface  8 (    dyke_2): Δλ̄=+1.742  Δμ̄=+1.782  Δρ̄=+0.074
  iface  9 (    dyke_3): Δλ̄=+2.088  Δμ̄=+2.090  Δρ̄=+0.083
  iface 10 (    dyke_4): Δλ̄=+1.999  Δμ̄=+1.937  Δρ̄=+0.069
  iface 11 (  gabbro_1): Δλ̄=+2.778  Δμ̄=+2.793  Δρ̄=+0.078
  iface 12 (  gabbro_2): Δλ̄=+2.865  Δμ̄=+2.800  Δρ̄=+0.075
  iface 13 (  gabbro_3): Δλ̄=+3.106  Δμ̄=+2.977  Δρ̄=+0.079
  iface 14 (  gabbro_4): Δλ̄=+3.084  Δμ̄=+2.921  Δρ̄=+0.074
  iface 15 (  gabb

## 3. Per-Layer Inner Block GMRES (Heterogeneous Foldy-Lax)

For each elastic layer we run the frequency-batched heterogeneous
block Foldy-Lax solver on the 32×32×1 voxel lattice and accumulate
a composite `(F, 9, 9)` tensor.  Because the block GMRES Krylov
basis scales as `F · n_cubes · k_rhs · max_iter`, we split the 256
positive frequencies into chunks of 64 (`freq_batch_size=64`) so each
individual solve stays comfortably under the 4 GB per-call cap.


In [4]:
dw = 2.0 * np.pi / config.T_record
omegas_real = np.arange(1, config.nw + 1, dtype=np.float64) * dw
omegas_complex = omegas_real + 1j * config.damping

freqs_hz = omegas_real / (2.0 * np.pi)
print(f"ω grid: {config.nw} positive frequencies")
print(f"  dω = 2π/T = {dw:.4f} rad/s")
print(f"  f range = [{freqs_hz[0]:.3f}, {freqs_hz[-1]:.3f}] Hz")
print(f"  damping = {config.damping:g} rad/s (imag part of ω)")

ω grid: 256 positive frequencies
  dω = 2π/T = 0.1963 rad/s
  f range = [0.031, 8.000] Hz
  damping = 0.1 rad/s (imag part of ω)


In [5]:
print("Per-layer heterogeneous block Foldy-Lax solves...\n")
t0 = time.perf_counter()
tmatrices, iters, rel_res = compute_layer_composite_tmatrices_het(
    config=config,
    voxel_contrasts=voxel_contrasts,
    omegas=omegas_complex,
    freq_batch_size=64,
    verbose=True,
)
inner_elapsed = time.perf_counter() - t0

max_res_all = max(float(np.max(r)) for r in rel_res.values())
print(f"\ninner solves: {len(tmatrices)} layers, elapsed = {inner_elapsed:.1f} s")
print(f"max rel residual across all layers × ω = {max_res_all:.2e}")
assert max_res_all < 1e-6, (
    f"Block GMRES did not converge everywhere: max rel res = {max_res_all:.2e}"
)

Per-layer heterogeneous block Foldy-Lax solves...



  iface  1 (sediment_1): iters≤12, max rel res=4.51e-09


  iface  2 (sediment_2): iters≤ 8, max rel res=8.89e-09


  iface  3 (sediment_3): iters≤ 8, max rel res=8.12e-09


  iface  4 (  pillow_1): iters≤ 7, max rel res=2.57e-09


...

  iface 14 (  gabbro_4): iters≤ 7, max rel res=3.69e-09


  iface 15 (  gabbro_5): iters≤ 7, max rel res=3.15e-09


  iface 16 (  gabbro_6): iters≤ 7, max rel res=2.65e-09


  iface 17 (  gabbro_7): iters≤ 7, max rel res=2.60e-09


  iface 18 (  gabbro_8): iters≤ 7, max rel res=1.60e-09

inner solves: 18 layers, elapsed = 3523.1 s
max rel residual across all layers × ω = 8.89e-09


## 4. Outer Riccati Interlayer-MS Sweep → R(ω, p)

Per-ω call to `interlayer_ms_reflectivity_9x9` on the 20-layer model
with the 18 composite T-matrices.  Returns both `R_total` (includes
scatterers) and `R_ref` (background stratified response).


In [6]:
p_values = np.linspace(config.p_min, config.p_max, config.n_p)

# Each composite T returned by the heterogeneous inner solver is the
# aggregate response of the entire M×M block of cubes (not of a single
# voxel), so the correct areal number density is one block per block
# area = 1 / (M · 2a)².  Equivalently this is the per-cube
# space-filling density 1/(2a)² divided by M² = n_cubes-per-block.
block_edge_km = config.voxels.M * 2.0 * config.voxels.a_km
n_density = 1.0 / block_edge_km**2
print(f"block edge  = {block_edge_km * 1000:.1f} m, area = {block_edge_km**2:.4f} km²")
print(f"n_density   = {n_density:.4f} blocks/km²")

t0 = time.perf_counter()
R_total_wp, R_ref_wp = compute_R_omega_p_stack(
    model=model,
    tmatrices_per_iface=tmatrices,
    omegas=omegas_complex,
    p_values=p_values,
    n_density=n_density,
)
outer_elapsed = time.perf_counter() - t0

print(
    f"outer sweep: {config.nw} frequencies × {config.n_p} slownesses, "
    f"elapsed = {outer_elapsed:.1f} s"
)
print(f"R_total shape = {R_total_wp.shape}")
print(f"R_ref   shape = {R_ref_wp.shape}")
print(f"max |R_total| = {np.abs(R_total_wp).max():.3f}")
print(f"max |R_ref|   = {np.abs(R_ref_wp).max():.3f}")
assert np.all(np.isfinite(R_total_wp)), "R_total has non-finite entries"
assert np.all(np.isfinite(R_ref_wp)), "R_ref has non-finite entries"

rel_pert = float(
    np.linalg.norm(R_total_wp - R_ref_wp) / max(np.linalg.norm(R_ref_wp), 1e-30)
)
print(f"relative perturbation ||R_total - R_ref|| / ||R_ref|| = {rel_pert:.3e}")

block edge  = 640.0 m, area = 0.4096 km²
n_density   = 2.4414 blocks/km²


outer sweep: 256 frequencies × 48 slownesses, elapsed = 418.4 s
R_total shape = (256, 48)
R_ref   shape = (256, 48)
max |R_total| = 672.036
max |R_ref|   = 0.308
relative perturbation ||R_total - R_ref|| / ||R_ref|| = 4.061e+01


## 5. Ricker × Hermitian IFFT → τ-p Traces


In [7]:
time_axis, traces_total = taup_traces_from_R_omega_p(
    R_omega_p=R_total_wp,
    omegas_real=omegas_real,
    T_record=config.T_record,
)
_, traces_ref = taup_traces_from_R_omega_p(
    R_omega_p=R_ref_wp,
    omegas_real=omegas_real,
    T_record=config.T_record,
)
print(
    f"time axis: {time_axis.shape[0]} samples, "
    f"dt={time_axis[1] - time_axis[0]:.4f} s, "
    f"T={time_axis[-1] + (time_axis[1] - time_axis[0]):.2f} s"
)
print(f"traces_total shape = {traces_total.shape}")
print(f"traces_ref   shape = {traces_ref.shape}")
print(f"|traces_total| max = {np.abs(traces_total).max():.3e}")
print(f"|traces_ref  | max = {np.abs(traces_ref).max():.3e}")

time axis: 514 samples, dt=0.0623 s, T=32.00 s
traces_total shape = (514, 48)
traces_ref   shape = (514, 48)
|traces_total| max = 7.703e+01
|traces_ref  | max = 1.059e+01


## 6. Display: Reference + Total Wiggle Gathers, $|R(\omega, p)|$ Heatmap


In [ ]:
t_max = 16.0  # seconds shown
tmask = time_axis <= t_max
t_plot = time_axis[tmask]

fig, axes = plt.subplots(1, 2, figsize=(12, 8), sharey=True)


def _wiggle(ax, traces_2d, title):
    """Per-trace normalised variable-area wiggle plot.

    Because the heterogeneous gather has strong scattering poles that
    dominate a global peak normalisation, every trace is scaled by its
    own peak so low- and high-slowness events remain visible.
    """
    tr = traces_2d[tmask, :]
    peaks = np.abs(tr).max(axis=0).astype(np.float64)
    peaks[peaks == 0.0] = 1.0
    dp = float(p_values[1] - p_values[0]) if len(p_values) > 1 else 0.1
    for j, p in enumerate(p_values):
        sig = np.clip(tr[:, j] / peaks[j], -0.8, 0.8)
        ax.plot(p + dp * sig, t_plot, "k-", linewidth=0.5)
        ax.fill_betweenx(
            t_plot, p, p + dp * sig, where=(sig > 0), color="k", linewidth=0.0
        )
    ax.set_xlabel("slowness p (s/km)")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)


_wiggle(axes[0], traces_ref, "reference medium (layered, no scatterers)")
_wiggle(axes[1], traces_total, "full model (heterogeneous voxel scatterers)")
axes[0].set_ylabel("time (s)")
axes[0].set_ylim(t_plot.max(), t_plot.min())
fig.suptitle(
    "τ-p wiggle gathers — oceanic crust, 18 heterogeneous layers (per-trace norm)"
)
fig.tight_layout()
plt.show()

In [ ]:
import matplotlib.colors as mcolors

amplitude = np.abs(R_total_wp)
vmax = float(amplitude.max())
vmin = max(1.0e-3, float(amplitude[amplitude > 0].min()))
if vmax <= vmin:
    vmax = vmin * 10.0
norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(
    np.maximum(amplitude, vmin),
    aspect="auto",
    origin="lower",
    extent=(
        float(p_values.min()),
        float(p_values.max()),
        float(freqs_hz.min()),
        float(freqs_hz.max()),
    ),
    cmap="viridis",
    norm=norm,
)
ax.set_xlabel("slowness p (s/km)")
ax.set_ylabel("frequency (Hz)")
ax.set_title(r"$|R_{\mathrm{total}}(\omega, p)|$ (log scale)")
fig.colorbar(im, ax=ax, label=r"$|R(\omega, p)|$")
fig.tight_layout()
plt.show()

## 7. Save Outputs (NPZ + PDFs)


In [10]:
output_dir = project_root / config.output_dir
output_dir.mkdir(parents=True, exist_ok=True)

# Pack layer_contrasts as a structured object array so downstream code
# can recover Δλ, Δμ, Δρ per voxel per iface.
layer_contrasts_serialised: dict[str, np.ndarray] = {}
for iface, cs in voxel_contrasts.items():
    arr = np.array([[c.Dlambda, c.Dmu, c.Drho] for c in cs], dtype=np.float64)
    layer_contrasts_serialised[f"iface_{iface:02d}"] = arr

npz_path = output_dir / config.npz_name
np.savez_compressed(
    npz_path,
    omegas_real=omegas_real,
    omegas_complex=omegas_complex,
    p_values=p_values,
    R_total_wp=R_total_wp,
    R_ref_wp=R_ref_wp,
    time=time_axis,
    traces_total=traces_total,
    traces_ref=traces_ref,
    rng_seed=np.int64(config.random_seed),
    config_yaml_raw=np.array(config.raw_yaml),
    **layer_contrasts_serialised,
)
print(f"NPZ saved to {npz_path}  ({npz_path.stat().st_size / 1e6:.2f} MB)")

plot_taup_wiggle_gather(
    time=time_axis,
    traces=traces_ref,
    p_values=p_values,
    title="Reference medium (no scatterers) — τ-p wiggle gather",
    output=output_dir / config.fig_wiggle_reference,
    t_max=t_max,
)
plot_taup_wiggle_gather(
    time=time_axis,
    traces=traces_total,
    p_values=p_values,
    title="Heterogeneous voxels — τ-p wiggle gather",
    output=output_dir / config.fig_wiggle_total,
    t_max=t_max,
)
plot_r_omega_p_heatmap(
    R_omega_p=R_total_wp,
    freqs_hz=freqs_hz,
    p_values=p_values,
    title=r"$|R_{\mathrm{total}}(\omega, p)|$",
    output=output_dir / config.fig_r_omega_p,
)
print("saved figures:")
for name in (
    config.fig_wiggle_reference,
    config.fig_wiggle_total,
    config.fig_r_omega_p,
):
    p = output_dir / name
    print(f"  {p}  ({p.stat().st_size / 1e3:.1f} kB)")

NPZ saved to /Users/tod/Desktop/SeismicInversion/GlobalMatrix/output/taup_scale_demo/taup_scale_demo.npz  (1.20 MB)


saved figures:
  /Users/tod/Desktop/SeismicInversion/GlobalMatrix/output/taup_scale_demo/taup_reference_wiggle_gather.pdf  (110.6 kB)
  /Users/tod/Desktop/SeismicInversion/GlobalMatrix/output/taup_scale_demo/taup_wiggle_gather.pdf  (133.9 kB)
  /Users/tod/Desktop/SeismicInversion/GlobalMatrix/output/taup_scale_demo/r_omega_p_heatmap.pdf  (46.5 kB)
